# T30B / E13 — Các lớp không tràn số của Sailor2-8B có cho giá trị đúng không

**Câu hỏi.** Với Qwen2.5-7B, T07 đã đối chiếu `float16` với `float32`: 27 lớp còn lại lệch trung bình
0,07 % thang đo. Với Sailor2-8B chưa có con số tương đương, vì lượt đối chiếu hết bộ nhớ hai lần khi hai
mô hình cùng nằm trong một tiến trình.

**Cách làm.** Mỗi kiểu số chạy trong **một tiến trình riêng**, ghi kết quả ra tệp, rồi so hai tệp. Đối chiếu
với `bfloat16` (cùng dải giá trị với `float32`, tốn bộ nhớ như `float16`). Vì `bfloat16` có ít chữ số có nghĩa
hơn, Qwen2.5-7B cũng được chạy theo đúng cách này để biết mức lệch nào là bình thường.

**Cách đọc.** Nếu độ lệch của Sailor2 cùng cỡ với Qwen2.5-7B thì các lớp còn lại dùng được; nếu lớn hơn
nhiều thì kết quả E13 phải kèm cảnh báo.

**Dự đoán ghi trước khi chạy:** độ lệch trung bình của Sailor2 không quá hai lần của Qwen2.5-7B.

**Chạy trong một phiên T4 riêng.** Ước tính 1–1,5 giờ, phần lớn là tải mô hình.

In [ ]:
# Ô 1 — lấy mã nguồn. Chạy lại được nhiều lần.
import subprocess
from pathlib import Path

REPO_URL = "https://github.com/wsunicorn/vihallulens.git"
REPO_DIR = Path("/kaggle/working/vihallulens")


def run(*args, cwd=None):
    done = subprocess.run(args, cwd=cwd, capture_output=True, text=True)
    if done.returncode:
        raise RuntimeError(" ".join(args) + chr(10) + done.stdout + done.stderr)
    return done.stdout.strip()


if (REPO_DIR / ".git").is_dir():
    run("git", "fetch", "--quiet", "origin", cwd=REPO_DIR)
    run("git", "reset", "--quiet", "--hard", "origin/main", cwd=REPO_DIR)
else:
    run("git", "clone", "--quiet", REPO_URL, str(REPO_DIR))

%cd /kaggle/working/vihallulens
print("commit:", run("git", "log", "--oneline", "-1", cwd=REPO_DIR))

In [ ]:
# Ô 2 — cài đặt, ghim phiên bản theo uv.lock.
!pip install -q --no-deps -e .
!pip install -q -r requirements-kaggle.txt

In [ ]:
# Ô 3 — TIỀN KIỂM. Vài giây, chạy trước mọi thứ.
import importlib.util
import sys
from pathlib import Path

import torch

sys.path.insert(0, "src")
from vihallulens.data.paths import find_raw_dir

problems = []
absent = [n for n in ("torch", "transformers", "bitsandbytes", "accelerate", "pandas")
          if importlib.util.find_spec(n) is None]
if absent:
    problems.append(f"thiếu gói {absent}")
if not torch.cuda.is_available():
    problems.append("chưa bật GPU T4")
else:
    print("GPU:", torch.cuda.get_device_name(0))
try:
    raw = Path(find_raw_dir())
    missing = [n for n in ['vihallu_train.csv', 'isedsc01_train.json'] if not (raw / n).is_file()]
    print("dữ liệu thô:", raw, "— thiếu", missing or "không")
    if missing:
        problems.append(f"thiếu tệp {missing}")
except Exception as error:
    problems.append(f"chưa gắn dataset dữ liệu thô ({error})")
if problems:
    raise SystemExit("TIỀN KIỂM HỎNG: " + "; ".join(problems))
print("Tiền kiểm đạt.")

In [ ]:
# Ô 4 — kiểm môi trường. Một phút, CPU.
!python scripts/probe_env.py
!python -m pytest tests/test_compare_dtypes.py tests/test_attention_hook.py tests/test_revisions.py -q
!df -h /root | tail -1

In [ ]:
# Ô 5 — bốn lượt, mỗi lượt một tiến trình. Một lượt hỏng KHÔNG dừng các lượt sau.
# Trọng số Sailor2 được xóa khỏi đĩa của phiên Kaggle sau khi dùng, để đủ chỗ tải Qwen2.5-7B.
import os
import shutil
import sys
from pathlib import Path

os.environ["PYTORCH_ALLOC_CONF"] = "expandable_segments:True"
OUT = Path("/kaggle/working/t30b")
OUT.mkdir(exist_ok=True)
MODELS = {"sailor2": "sail/Sailor2-8B-SFT", "qwen7b": "Qwen/Qwen2.5-7B-Instruct"}
status = {}
for key, model in MODELS.items():
    for dtype in ("float16", "bfloat16"):
        target = OUT / f"{key}_{dtype}.npz"
        if target.is_file():
            status[key, dtype] = "đã có"
            continue
        print("=" * 78)
        print(f"  {model} ở {dtype}", flush=True)
        done = subprocess.run([sys.executable, "scripts/compare_dtypes.py", "--model", model,
                               "--dtype", dtype, "--save", str(target)],
                              capture_output=True, text=True)
        print(done.stdout[-3500:])
        if done.returncode:
            print(done.stderr[-2500:])
        status[key, dtype] = "xong" if target.is_file() else f"HỎNG (mã {done.returncode})"
    hub = Path.home() / ".cache/huggingface/hub" / ("models--" + model.replace("/", "--"))
    if key == "sailor2" and all((OUT / f"{key}_{d}.npz").is_file()
                                for d in ("float16", "bfloat16")):
        shutil.rmtree(hub, ignore_errors=True)
for (key, dtype), value in status.items():
    print(f"  {key:<8} {dtype:<9} {value}")

In [ ]:
# Ô 6 — so sánh. Cần đủ hai tệp của một mô hình; thiếu thì báo và bỏ qua mô hình đó.
import subprocess
import sys
from pathlib import Path

OUT = Path("/kaggle/working/t30b")
for key in ("qwen7b", "sailor2"):
    low, high = OUT / f"{key}_float16.npz", OUT / f"{key}_bfloat16.npz"
    print("=" * 78)
    if not (low.is_file() and high.is_file()):
        print(f"  {key}: thiếu tệp, không so được")
        continue
    done = subprocess.run([sys.executable, "scripts/compare_dtypes.py", "--from-saved",
                           str(low), str(high)], capture_output=True, text=True)
    print(done.stdout or done.stderr)
    (OUT / f"{key}_so_sanh.txt").write_text(done.stdout, encoding="utf-8")
!ls -la /kaggle/working/t30b

In [ ]:
# Ô 7 — gói lại để tải về một tệp: /kaggle/working/t30b.zip
import shutil

print(shutil.make_archive("/kaggle/working/t30b", "zip", "/kaggle/working/t30b"))